In [46]:
from pathlib import Path

from helpers.embedding_experiment import (
    build_cli_command,
    get_model_info,
    list_models,
    load_config,
    run_experiment,
    setup_experiment,
)

In [47]:
list_models()

Family          Script                                        Description
----------------------------------------------------------------------------------------------------
dinov3          v4_dino_embeddings_lancedb.py                 DINOv2/v3 ViT models with patch embeddings and attention maps


In [ ]:
PROJECT_ROOT = Path("/Users/ncheruku/Documents/Work/github/bams-ai-data-exploration/.claude/worktrees/vigilant-nobel")

SOURCE_URI = PROJECT_ROOT / "data" / "lancedb" / "shared_source"
IMG_RAW_TBL_NAME = "era5_sample_images"
DB_URI = PROJECT_ROOT / "data" / "lancedb" / "experiments" / "era5"
AUTHOR = "Cherukuru. N. W"

TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [ ]:
# Model-specific config — change this for a different model family
PROJECT_NAME = "dinov3"
model_info = get_model_info(PROJECT_NAME)

MODEL = model_info["default_model"]
SCRIPT = model_info["script_path"]
BATCH = model_info["default_batch"]
WORKERS = model_info["default_workers"]
SCAN_BATCH = model_info.get("default_scan_batch", 1000)

print(f"Family: {PROJECT_NAME}")
print(f"Script: {model_info['script']}")
print(f"Model:  {MODEL}")

In [ ]:
experiment = setup_experiment(
    PROJECT_NAME, AUTHOR, SOURCE_URI, IMG_RAW_TBL_NAME,
    DB_URI, project_root=PROJECT_ROOT,
)

In [ ]:
# Case 1: Run inline (interactive notebook workflow)
run_experiment(
    SCRIPT, SOURCE_URI, IMG_RAW_TBL_NAME,
    DB_URI, experiment["config_name"], PROJECT_NAME,
    MODEL, batch=BATCH, scan_batch=SCAN_BATCH, workers=WORKERS,
)

In [ ]:
# Case 2: Build CLI command for PBS / external job submission
cmd = build_cli_command(
    SCRIPT, SOURCE_URI, IMG_RAW_TBL_NAME,
    DB_URI, experiment["config_name"], PROJECT_NAME,
    MODEL, batch=BATCH, scan_batch=SCAN_BATCH, workers=WORKERS,
)
print(cmd)

In [ ]:
# Inspect config after run completes (works for both Case 1 and Case 2)
config = load_config(DB_URI, experiment["config_name"])
config

In [ ]:
import os


def dir_size_bytes(path: Path) -> int:
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += (Path(root) / f).stat().st_size
    return total


# table_path = db_dir / "era5_sample_images.lance"


size_bytes = dir_size_bytes("/glade/work/ncheruku/research/bams-ai-data-exploration/data/lancedb/experiments/era5/dinov3_patch_embeddings.lance")

# size_bytes = dir_size_bytes(table_path)


print(f"{size_bytes / 1024**2:.2f} MB")

In [ ]:
import lancedb

db = lancedb.connect(str(DB_URI))
patch_tbl = db.open_table(experiment["patch_emb_name"])

In [ ]:
patch_tbl.schema

In [ ]:
row = patch_tbl.search().limit(1).to_pandas().iloc[0]
row["image_id"]

In [ ]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"Is CUDA available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")

In [ ]:
patch_tbl.create_index(metric="cosine", index_type="IVF_PQ", num_partitions=128, num_sub_vectors=96, accelerator="cuda", vector_column_name="embedding")